# 03 · 多表连接 JOIN（SQL 真正的威力所在）

> 规范化把数据拆到了 5 张表里，但你做分析时想看的是“一张拼好的宽表”。JOIN 就是把这些分散的表按主键/外键重新拼起来的操作。
> 你在 Pandas 里用过 `pd.merge(left, right, on=...)`，SQL 的 JOIN 与之几乎一一对应。这一课是整个 SQL 最核心、也最容易写出“看似跑通其实数字错了”的地方——**重点不是语法，是搞清楚每种 JOIN 保留哪些行、以及连完为什么会“行数膨胀”**。

## 本课地图
1. 为什么规范化逼着你必须 JOIN
2. INNER JOIN：只保留两边都匹配的行
3. LEFT / RIGHT / FULL JOIN：保留“没匹配上”的那一侧
4. ON vs WHERE：连接条件和过滤条件别写混
5. 多表连接、自连接（员工-经理）、USING
6. 最大的坑：JOIN 后行数膨胀与聚合重复计数

In [1]:
# 统一环境：连接数据库（绝对路径，可反复运行）
import sqlite3
import pandas as pd

# 两个库都用绝对路径写死，避免“相对路径在哪个工作目录下”的坑
SHOP_DB = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\shop.db"
SS_DB  = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\superstore.db"

# 让 DataFrame 打印好看一点（你已经熟悉 Pandas，这里直接拿它当 SQL 的“显示层”）
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

# 教学默认连 shop.db；需要 superstore 时再切
conn = sqlite3.connect(SHOP_DB)

def q(sql, conn=conn):
    """执行一条 SQL，把结果表直接返回成 Pandas DataFrame。"""
    return pd.read_sql_query(sql, conn)

print("已连接 shop.db。可用 q('SELECT ...') 跑查询。")
print("shop.db 表：", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])

已连接 shop.db。可用 q('SELECT ...') 跑查询。
shop.db 表： ['customers', 'employees', 'order_items', 'orders', 'products']


## 1. 为什么规范化逼着你必须 JOIN

回顾 00 课：客户信息在 `customers`、商品在 `products`、订单在 `orders`、买了啥在 `order_items`。现在老板问：“每笔订单里，客户叫啥、买了什么商品、多少钱？”——这条信息**分散在四张表里**，你必须把它们拼起来。

In [2]:
# 先看两张表各自有啥：orders 只有 customer_id，没有客户姓名
q('''SELECT order_id, customer_id, order_date, ship_mode FROM orders LIMIT 3''')

,order_id,customer_id,order_date,ship_mode
0,1,162,2025-06-15,二级配送
1,2,153,2025-02-10,当天达
2,3,74,2025-09-04,标准配送


In [3]:
# customers 里才有姓名和地区。想在订单旁显示客户名，就得按 customer_id 拼起来
q('''SELECT customer_id, name, region FROM customers WHERE customer_id IN (1,2,3)''')

,customer_id,name,region
0,1,许军,华东
1,2,马欣怡,华东
2,3,郭秀英,华北


## 2. INNER JOIN：只保留两边都匹配的行

`A INNER JOIN B ON A.key = B.key`：取两表“键相等”的行，**任何一边没匹配上的行都丢弃**。等价 Pandas `pd.merge(a, b, how='inner')`。

In [4]:
# 把订单和客户按 customer_id 拼起来：只保留“有客户信息的订单”
q('''
SELECT o.order_id, o.order_date,
       c.name AS customer_name, c.region AS customer_region
FROM orders o
INNER JOIN customers c ON c.customer_id = o.customer_id
LIMIT 8
''')

,order_id,order_date,customer_name,customer_region
0,1,2025-06-15,谢涛,华南
1,2,2025-02-10,林勇,西南
2,3,2025-09-04,朱平,华南
3,4,2022-09-06,郭雨辰,华中
4,5,2025-04-17,吴超,华南
5,6,2024-07-13,林洋,华北
6,7,2025-03-25,赵勇,华中
7,8,2023-04-06,赵勇,华中


> 注意：`INNER` 通常省略，直接写 `JOIN` 默认就是 INNER。`ON` 后面是“用哪一列连”，类似 Pandas 的 `on=`。表别名 `o` / `c` 让 `o.customer_id`、`c.customer_id` 一眼分清来源——**两表都有 customer_id 时，不写表前缀会报“列名不明确”**。

## 3. LEFT / RIGHT / FULL JOIN：保留没匹配上的那一侧

INNER 会丢掉“孤立”的行。现实里你常想保留某一侧全部行，没匹配上的另一侧填 NULL：

| JOIN 类型 | 保留什么 | Pandas how= |
|---|---|---|
| `INNER JOIN` | 两表都匹配的行（交集） | `inner` |
| `LEFT JOIN` | 左表全部 + 右表匹配的，没匹配填 NULL | `left` |
| `RIGHT JOIN` | 右表全部 + 左表匹配的 | `right` |
| `FULL JOIN` | 两表全部，缺哪边补哪边 NULL | `outer` |
| `CROSS JOIN` | 两表所有行两两组合（笛卡尔积） | 少见 |

In [5]:
# 自连接演示 LEFT JOIN：员工 e 连自己的经理 m
# 大老板张总没有上级(m.manager_id IS NULL)，LEFT JOIN 会把他也保留，m 那一列是 NULL
q('''
SELECT e.employee_id, e.name AS 员工, e.title,
       m.name AS 经理, m.title AS 经理职位
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.employee_id
ORDER BY e.employee_id
LIMIT 10
''')

,employee_id,员工,title,经理,经理职位
0,1,张总,总经理,None,None
1,2,销售总监,区域总监,张总,总经理
2,3,华东销售经理,销售经理,销售总监,区域总监
3,4,华北销售经理,销售经理,销售总监,区域总监
4,5,华南销售经理,销售经理,销售总监,区域总监
5,6,西南销售经理,销售经理,销售总监,区域总监
6,7,华中销售经理,销售经理,销售总监,区域总监
7,8,高刚,销售代表,华东销售经理,销售经理
8,9,杨芳,销售代表,华北销售经理,销售经理
9,10,林杰,销售代表,华北销售经理,销售经理


上面这句叫**自连接 (self join)**：一张表 `employees` 同时当“员工表 e”和“经理表 m”用。因为经理本身也是员工，存在同一张表里、靠 `manager_id` 指回自己。这是典型的树形/层级数据。

In [6]:
# 用 LEFT JOIN + IS NULL 找“没有经理的人”= 大老板
# （INNER JOIN 会把张总丢掉，因为他在经理侧匹配不到）
q('''
SELECT e.employee_id, e.name, e.title
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.employee_id
WHERE m.employee_id IS NULL
''')

,employee_id,name,title
0,1,张总,总经理


> **“找没有对应记录的行”是 LEFT JOIN 的经典用途**：左表 LEFT JOIN 右表，再 `WHERE 右表.主键 IS NULL`，就得到“左边有、右边没有”的差集。这是 04 课集合操作 EXCEPT 的另一种写法。

In [7]:
# RIGHT JOIN / FULL JOIN 在 SQLite 里：
# SQLite 早期不支持 RIGHT / FULL OUTER JOIN（8.x 才补 FULL）。
# 但思想一样，这里用 LEFT JOIN 的翻转来理解 RIGHT：
# 把两张表换个顺序，LEFT JOIN 就等价于原来的 RIGHT JOIN。
# CROSS JOIN 演示：两列小表两两组合
q('''
WITH a(x) AS (SELECT 1 UNION ALL SELECT 2),
     b(y) AS (SELECT 'A' UNION ALL SELECT 'B')
SELECT * FROM a CROSS JOIN b
''')

,x,y
0,1,A
1,1,B
2,2,A
3,2,B


## 4. ON vs WHERE：连接条件和过滤条件别写混

新手常把“过滤条件”也塞进 ON，结果行为完全不同（尤其 LEFT JOIN 时）：

- `ON` 决定“两张表怎么配对”；
- `WHERE` 在配对完之后“再筛结果行”。

对 LEFT JOIN，把对右表的过滤写进 ON，和写进 WHERE，结果差很多：

In [8]:
# 写法对比：同样想看“华东员工及其经理”
# 注意 LEFT JOIN 时，右表条件放 ON 还是 WHERE 的区别
q('''
SELECT e.name AS 员工, e.region, m.name AS 经理
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.employee_id AND e.region = '华东'
ORDER BY e.employee_id
LIMIT 12
''')

,员工,region,经理
0,张总,全国,None
1,销售总监,全国,None
2,华东销售经理,华东,销售总监
3,华北销售经理,华北,None
4,华南销售经理,华南,None
5,西南销售经理,西南,None
6,华中销售经理,华中,None
7,高刚,华东,华东销售经理
8,杨芳,华北,None
9,林杰,华北,None


> 直觉：**INNER JOIN 时 ON/WHERE 写过滤结果差不多；一旦用 LEFT JOIN，对右表的过滤条件放 ON 还是 WHERE 就天差地别**。这是面试和实操的高频坑，遇到 LEFT JOIN 结果不对，第一反应检查这个。

## 5. 多表连接与 USING

真实分析常常一口气连 3~4 张表。

In [9]:
# 一张完整的“订单明细宽表”：订单 + 客户 + 商品 + 明细
q('''
SELECT o.order_id, o.order_date,
       c.name   AS 客户, c.segment,
       p.product_name, p.category,
       oi.quantity, oi.sales, oi.profit
FROM orders o
JOIN customers c    ON c.customer_id = o.customer_id
JOIN order_items oi ON oi.order_id   = o.order_id
JOIN products p     ON p.product_id  = oi.product_id
ORDER BY o.order_id
LIMIT 10
''')

,order_id,order_date,客户,segment,product_name,category,quantity,sales,profit
0,1,2025-06-15,谢涛,个人客户,电脑-04号,科技产品,6,43202.52,17984.04
1,1,2025-06-15,谢涛,个人客户,显示器-02号,科技产品,3,4107.96,153.09
2,2,2025-02-10,林勇,个人客户,床-04号,家具,4,5033.88,747.04
3,2,2025-02-10,林勇,个人客户,标签-02号,办公用品,1,24.05,3.50
4,2,2025-02-10,林勇,个人客户,标签-04号,办公用品,6,132.19,19.21
5,3,2025-09-04,朱平,个人客户,耳机-04号,科技产品,6,1316.45,-415.27
6,3,2025-09-04,朱平,个人客户,耳机-03号,科技产品,1,524.68,183.04
7,3,2025-09-04,朱平,个人客户,平板-04号,科技产品,1,1124.18,13.10
8,4,2022-09-06,郭雨辰,个人客户,沙发-04号,家具,6,9492.44,2027.54
9,4,2022-09-06,郭雨辰,个人客户,桌子-01号,家具,4,3969.12,1024.48


`USING(列名)` 是简写：当两表连接键**同名**时，`ON a.id = b.id` 可写成 `USING(id)`，而且结果里这个键只出现一列，不重复。

In [10]:
# USING 简写（演示：键同名时）
q('''
SELECT order_id, customers.customer_id, name
FROM orders
JOIN customers USING (customer_id)
LIMIT 5
''')

,order_id,customer_id,name
0,82,1,许军
1,212,1,许军
2,452,1,许军
3,641,1,许军
4,684,1,许军


## 6. 最大的坑：JOIN 后行数膨胀与重复计数

这是“看似跑通、数字却翻倍”的头号原因，必须单独讲。

规范化后，`orders`(2600) 和 `order_items`(6407) 是一对多：一张订单有 N 条明细。连起来之后，**这张订单会在结果里重复出现 N 次**。

In [11]:
# 连起来后，orders 侧的列被“复制”到每条明细上
# orders 2600 行，order_items 6407 行 -> 连起来应是 6407 行（不是 2600，也不是 2600+6407）
q('''
SELECT COUNT(*) AS 连完后的行数
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
''')

,连完后的行数
0,6407


In [12]:
# 陷阱来了：如果你在这个膨胀后的结果上 COUNT(*) 想数“订单数”，会得到 6407 而不是 2600！
# 正确做法：数订单用 COUNT(DISTINCT o.order_id)
q('''
SELECT COUNT(*)                 AS 错误_数明细行数,
       COUNT(DISTINCT o.order_id) AS 正确_数订单数
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
''')

,错误_数明细行数,正确_数订单数
0,6407,2600


> 更隐蔽的双重求和：如果你 JOIN 了**两张都是一对多**的表（比如订单同时连“明细”和“支付流水”），行数会按两边倍数相乘，此时 `SUM(sales)` 会被放大。经验法则：
> 1. **连完先 `COUNT(*)` 看行数是否符合预期**；
> 2. 数“独立的订单/客户”永远用 `COUNT(DISTINCT 主键)`；
> 3. 求和之前想清楚：我连进来的这张表，是会让行数翻倍的吗？

下面验证一个正确的聚合：连表后按地区求和销售额，因为订单-明细是一对多、地区来自订单，每条明细都带了正确地区，求和是对的。

In [13]:
# 正确示范：连表后聚合（地区来自 orders，每条明细都正确归属一个地区）
q('''
SELECT o.region,
       SUM(oi.sales)               AS 总销售额,
       COUNT(DISTINCT o.order_id)  AS 订单数
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY o.region
ORDER BY 总销售额 DESC
''')

,region,总销售额,订单数
0,华东,6090607.00,606
1,华南,5873532.64,587
2,华中,5364460.57,563
3,西南,4051805.89,446
4,华北,3659920.73,398


## 动手小练习

1. 连接 `order_items` 和 `products`，查出每个商品的 `product_name`、总销量、总销售额，按销售额降序前 10。
2. 用自连接查出“每个销售代表(e.title='销售代表')的经理是谁”，列出员工名 + 经理名。
3. 用 LEFT JOIN + IS NULL 找出“没有任何订单的客户”（提示：本库所有客户都下过单，预期结果为空，理解为什么即可）。

In [14]:
# TODO(你): 在下面练习
# q(''' ... ''')

## 本课小结

- JOIN 是把规范化拆散的表按主外键拼回宽表，对应 Pandas merge。
- INNER 取交集；LEFT 保左表、RIGHT 保右表、FULL 保两边；CROSS 是笛卡尔积。
- 自连接 = 一张表连自己（员工-经理层级）。
- LEFT JOIN 时，右表过滤条件放 ON 还是 WHERE 结果不同，是高频坑。
- 连一对多会让行数膨胀；数独立实体用 `COUNT(DISTINCT 主键)`，求和前先想清楚会不会重复计数。

## 来源与延伸

- Mode SQL Tutorial · SQL JOINs（图解清晰）：https://mode.com/sql-tutorial/sql-joins/
- SQLBolt 第 6~7 课（多表查询）：https://sqlbolt.com/
- SQLite 官方 JOIN 语法（含 LEFT/CROSS/USING）：https://www.sqlite.org/lang_join.html
- SQLZoo · JOIN 操作：https://sqlzoo.net/wiki/JOIN
- LeetCode Database（中等题大量考多表连接）：https://leetcode.com/problemset/database/